In [27]:
'''
git clone https://github.com/Superzchen/iFeature
python iFeature/iFeature.py --file pHopt_sequences.fasta --type AAC --out baseline_features_aac.tsv
'''

'\ngit clone https://github.com/Superzchen/iFeature\npython iFeature/iFeature.py --file pHopt_sequences.fasta --type AAC --out baseline_features_aac.tsv\n'

In [28]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, max_error, mean_absolute_error

In [29]:
df = pd.read_csv("pHopt_data.csv")
print(df.head())

   Unnamed: 0   Accession                       Organism  EC Number  \
0           1      P21673                   Homo sapiens   2.3.1.57   
1           2      I3WU34     Exiguobacterium acetylicum  3.2.1.142   
2           3  A0A0B5JT51            Exiguobacterium sp.   3.2.1.41   
3           4  A0A1V0FWX7  Geobacillus thermocatenulatus   3.2.1.41   
4           5      P52902                  Pisum sativum  1.2.1.104   

   Sample Weight  pHopt     Split Test <20% to Train  \
0          0.366   7.47  Training                NaN   
1          0.366   6.00  Training                NaN   
2          0.366   8.50  Training                NaN   
3          0.366   6.50  Training                NaN   
4          0.366   7.60  Training                NaN   

                                            Sequence  
0  MAKFVIRPATAADCSDILRLIKELAKYEYMEEQVILTEKDLLEDGF...  
1  MKRIRSVCIVALTFALIFSGLSLSGQALEKGKSTLVIIHYKEDPNA...  
2  MNRLKSVCAVVLTFALIFSLFPVNSLALEKGKSTLVIIHYKEDKTS...  
3  MLHISRTFAAYLD

In [30]:
with open("pHopt_data.fasta", "w") as file:
    for index, row in df.iterrows():
        accession = row['Accession']
        sekvence = row['Sequence']
        
        file.write(f">{accession}\n") 
        file.write(f"{sekvence}\n")

In [31]:
df_fasta = pd.read_csv("pHopt_data.fasta")
print(df_fasta.head())


                                             >P21673
0  MAKFVIRPATAADCSDILRLIKELAKYEYMEEQVILTEKDLLEDGF...
1                                            >I3WU34
2  MKRIRSVCIVALTFALIFSGLSLSGQALEKGKSTLVIIHYKEDPNA...
3                                        >A0A0B5JT51
4  MNRLKSVCAVVLTFALIFSLFPVNSLALEKGKSTLVIIHYKEDKTS...


In [32]:
X_baseline = pd.read_csv('pHopt_ifeatures.tsv', sep='\t', index_col=0)
df_original = pd.read_csv('pHopt_data.csv', index_col='Accession') 

In [33]:
train_mask = df_original['Split'].isin(['Training', 'Validation'])
test_mask = df_original['Split'] == 'Testing'

train_accessions = df_original[train_mask].index
test_accessions = df_original[test_mask].index

X_train_baseline = X_baseline.loc[X_baseline.index.isin(train_accessions)]
X_test_baseline = X_baseline.loc[X_baseline.index.isin(test_accessions)]

y_train_baseline = df_original.loc[X_train_baseline.index, 'pHopt']
y_test_baseline = df_original.loc[X_test_baseline.index, 'pHopt']

In [34]:
pipeline_baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf'))
])

param_grid_baseline = {
    'svr__C': [1, 2, 3, 5],
    'svr__epsilon': [0.1, 0.3, 0.5],
    'svr__gamma': ['scale', 0.1, 0.01]
}

grid_search_baseline = GridSearchCV(
    estimator=pipeline_baseline, 
    param_grid=param_grid_baseline, 
    cv=5, 
    scoring='neg_mean_absolute_error', 
    n_jobs=-1
)

grid_search_baseline.fit(X_train_baseline, y_train_baseline)
best_baseline_svr = grid_search_baseline.best_estimator_
best_baseline_svr = grid_search_baseline.best_estimator_
y_pred_baseline = best_baseline_svr.predict(X_test_baseline)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

In [37]:
mse_base = mean_squared_error(y_test_baseline, y_pred_baseline)
rmse_base = np.sqrt(mse_base)
mae_base = mean_absolute_error(y_test_baseline, y_pred_baseline)
max_err_base = max_error(y_test_baseline, y_pred_baseline)
r2_base = r2_score(y_test_baseline, y_pred_baseline)

r2_train_base = r2_score(y_train_baseline, best_baseline_svr.predict(X_train_baseline))

print(f"Best params SVR       : {grid_search_baseline.best_params_}")
print(f"MSE                   : {mse_base:.3f}")
print(f"RMSE                  : {rmse_base:.3f}")
print(f"MAE                   : {mae_base:.3f}")
print(f"Max error             : {max_err_base:.3f}")
print(f"R^2 (Test)            : {r2_base:.3f}")
print(f"R^2 (Train)           : {r2_train_base:.3f}")

Best params SVR       : {'svr__C': 1, 'svr__epsilon': 0.1, 'svr__gamma': 'scale'}
MSE                   : 0.870
RMSE                  : 0.933
MAE                   : 0.665
Max error             : 4.986
R^2 (Test)            : 0.347
R^2 (Train)           : 0.675


In [39]:
import subprocess
import os

fasta_file = "ligasa.fasta"
feature_types = ["AAC", "DPC", "CTDC", "CTDT", "CTDD"]
dataframes = []

for f_type in feature_types:
    out_file = f"temp_{f_type}.tsv"

    command = f"python iFeature/iFeature.py --file {fasta_file} --type {f_type} --out {out_file}"
    subprocess.run(command, shell=True, capture_output=True)
    if os.path.exists(out_file):
        df = pd.read_csv(out_file, sep='\t', index_col=0)
        dataframes.append(df)
        os.remove(out_file)
    else:
        print(f"❌") 


super_baseline_df = pd.concat(dataframes, axis=1)

final_out = "ligasa_ifeatures.tsv"
super_baseline_df.to_csv(final_out, sep='\t')

In [40]:
nova_data_aac = pd.read_csv('ligasa_ifeatures.tsv', sep='\t', index_col=0)

odhadnute_pH = best_baseline_svr.predict(nova_data_aac)
print(f"Predikované pH: {odhadnute_pH[0]:.2f}")

Predikované pH: 7.20


In [42]:
import subprocess
import os

fasta_file = "atpasa.fasta"
feature_types = ["AAC", "DPC", "CTDC", "CTDT", "CTDD"]
dataframes = []

for f_type in feature_types:
    out_file = f"temp_{f_type}.tsv"

    command = f"python iFeature/iFeature.py --file {fasta_file} --type {f_type} --out {out_file}"
    subprocess.run(command, shell=True, capture_output=True)
    if os.path.exists(out_file):
        df = pd.read_csv(out_file, sep='\t', index_col=0)
        dataframes.append(df)
        os.remove(out_file)
    else:
        print(f"❌") 


super_baseline_df = pd.concat(dataframes, axis=1)

final_out = "atpasa_ifeatures.tsv"
super_baseline_df.to_csv(final_out, sep='\t')

In [43]:
nova_data_aac = pd.read_csv('atpasa_ifeatures.tsv', sep='\t', index_col=0)

odhadnute_pH = best_baseline_svr.predict(nova_data_aac)
print(f"Predikované pH: {odhadnute_pH[0]:.2f}")

Predikované pH: 7.51
